# Creating cohort for all patients

File is commented out because it creates a cohort in the All of Us open research access dataset. If you have an account, you can uncomment the code to run and duplicate our cohort. 

In [ ]:
'''
from IPython.display import display, HTML
import pandas as pd
import numpy as np, scipy
import matplotlib, matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import date
import seaborn as sns
import os
import subprocess
import pickle
from sklearn.model_selection import train_test_split
'''

In [ ]:
'''
print(os.getcwd())
'''

In [ ]:
#version = %env WORKSPACE_CDR

## Pregnancy starts

In [ ]:
'''
preg_sql = """
    SELECT
        c_occurrence.condition_concept_id
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    4299535
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )
            ) c_occurrence 
        """

preg_df = pd.read_gbq(
    preg_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

preg_df.head(5)
print(preg_df.shape)
'''

In [ ]:
'''
print(preg_df.head(15))
'''

In [ ]:
'''
preg_df['condition_start_datetime'] = pd.to_datetime(preg_df['condition_start_datetime'])

preg_df = preg_df.sort_values(by=['person_id', 'condition_start_datetime'])

filtered_rows = []
for i in range(len(preg_df)):
    if i == 0:
        # Keep earliest preg date for every patient
        filtered_rows.append(preg_df.iloc[i])
    else:
        #only do further calculations if the patient is the same
        if preg_df.iloc[i]['person_id'] == preg_df.iloc[i-1]['person_id']:
            #check differences in pregnancy dates
            date_diff = (preg_df.iloc[i]['condition_start_datetime'] - preg_df.iloc[i-1]['condition_start_datetime']).days
            if date_diff > 365:
                #if the start dates are more than a year apart, then it's different pregnancies
                filtered_rows.append(preg_df.iloc[i])
        else:
            filtered_rows.append(preg_df.iloc[i])


filtered_df = pd.DataFrame(filtered_rows)
filtered_df = filtered_df.reset_index(drop=True)
print(filtered_df.shape)
'''

In [ ]:
'''
print(filtered_df.head())
'''

## Deliveries

In [ ]:
'''
##Look at different types of deliveries.
##Connect deliveries with pregnancy starts for final cohort
##Set start and end dates for pregnancies --> end date after start and less than one year after start

#Normal delivery, trauma during birth, heart rate anomoly, C-section, perineal tear 1 delivery, PPH,
#perineal tear 2 delivery, amniotic fluid delivery, premature delivery, preterm premature delivery
deliveries = [441641, 439093, 443247, 193277, 192978, 443929, 198499, 4063306, 4086393, 45757176]
'''

In [ ]:
'''
deliveries_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_occurrence.condition_start_datetime as delivery_date,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    441641, 439093, 443247, 193277, 192978, 443929, 198499, 4063306, 4086393, 45757176
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )
            ) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

deliveries_df = pd.read_gbq(
    deliveries_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(deliveries_df.head(5))
print(deliveries_df.shape)
'''

In [ ]:
'''
deliveries_df['end_date'] = pd.to_datetime(deliveries_df['delivery_date'])
deliveries_df = deliveries_df.sort_values(by=['person_id', 'delivery_date'])

merged_df = filtered_df.merge(deliveries_df, on='person_id')
print(merged_df.shape)
merged_df = merged_df[(merged_df['delivery_date'] > merged_df['condition_start_datetime']) & 
                      (merged_df['delivery_date'] <= merged_df['condition_start_datetime'] + pd.DateOffset(days=365))]

# Sort by 'id' and 'end_date' to ensure the first match is kept
merged_df = merged_df.sort_values(by=['person_id', 'delivery_date'])

# Drop duplicates to keep the first match for each start date
result_df = merged_df.drop_duplicates(subset=['person_id', 'condition_start_datetime'], keep='first').reset_index(drop=True)

print(result_df)
'''

In [ ]:
'''
print(result_df.shape)
'''

In [ ]:
'''
df = result_df[['person_id', 'condition_concept_id_x', 'standard_concept_name_x',
               'condition_start_datetime', 'condition_concept_id_y', 'source_concept_name_y',
                'delivery_date']].copy()
'''                

In [ ]:
'''
df = df.rename(columns={"condition_concept_id_x": "preg_concept_id", "standard_concept_name_x": "preg_name",
                       'condition_start_datetime': 'preg_date', 'condition_concept_id_y': 'delivery_concept_id', 
                       'sourse_concept_name_y': 'delivery_name'})
'''

In [ ]:
'''
df["id"] = df.index
'''

In [ ]:
'''
df['censor_date'] = df['delivery_date'] - pd.DateOffset(weeks=40)
'''

In [ ]:
'''
print(df.head())
print(df.shape)
'''

In [ ]:
'''
df.to_pickle("./pregnancy_episodes_date.pkl")
'''